In [1]:
from llm_zoomcamp.homeworks.week_2_vector_search.embedder import Embedder

embed = Embedder()

Q1. Embedding a query

In [2]:
q = "How does approximate nearest neighbor search work??"

v = embed.encode(q)

In [3]:
v[0]

np.float64(-0.01016662288857993)

Loading the data

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
document = [doc for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md']

Q2. Cosine similarity

In [6]:
document = [doc for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'][0]

In [7]:
dv = embed.encode(document['content'])

In [8]:
v.dot(dv)

np.float64(0.36249401229570577)

Q3. Chunking and search by hand

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
import numpy as np

texts = []

for doc in chunks:
    text = doc['content']
    texts.append(text)

embed_chunks = embed.encode_batch(texts)

X = np.array(embed_chunks)

scores = X.dot(v)

In [11]:
idx = np.argmax(scores)
idx

np.int64(94)

In [12]:
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

Q4. Vector search with minsearch

In [13]:
from minsearch import VectorSearch

v_index = VectorSearch(keyword_fields=['filename'])

v_index.fit(X, chunks)

In [14]:
query = 'What metric do we use to evaluate a search engine?'
query_vector = embed.encode(query)

In [15]:
results = v_index.search(query_vector, num_results=5)

In [16]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

Q5. Text search vs vector search

In [17]:
query = 'How do I store vectors in PostgreSQL?'

In [18]:
from minsearch import Index

text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

text_results = text_index.search(query, num_results=5)

In [19]:
# Vector search
query_vector = embed.encode(query)

In [20]:
vector_results = v_index.search(query_vector, num_results=5)

In [21]:
[filename for filename in list(set([doc['filename'] for doc in vector_results])) if filename not in list(set([doc['filename'] for doc in text_results]))]

['02-vector-search/lessons/08-pgvector.md']

Q6. Hybrid search

In [22]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [23]:
query = 'How do I give the model access to tools?'

In [24]:
text_results = text_index.search(query, num_results=5)

query_vector = embed.encode(query)
vector_results = v_index.search(query_vector, num_results=5)

In [25]:
results = rrf([vector_results, text_results])

In [26]:
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca